In [ ]:
import os
os.chdir(os.path.expandvars("/scratch/vp91/$USER/intro-to-cuda"))

# Understanding Warps in CUDA


> **Overview**
>
> * **Time:** 30 min
> 
> 1. Learn about warps in CUDA programming.
> 1. Understand how warps affect performance and execution in CUDA kernels.

Warps are a fundamental concept in CUDA programming, representing how threads are organized and executed on NVIDIA GPUs. Understanding warps is crucial
for writing efficient CUDA kernels.

## What is a Warp?

A warp is a group of threads that execute the same instruction at the same time on a CUDA-enabled GPU.

* A **warp** consists of **32 threads** (in all current GPUs) that are executed **in parallel**.
* When a CUDA kernel is launched, threads are grouped into **blocks**, which are further divided into **warps** of 32 threads each.
* The 32 threads in a warp execute the **same instruction simultaneously** (SIMT - Single Instruction, Multiple Threads), but on **different data**.

## Why Use Warps?


Warps are designed to:

* Maximize **instruction-level parallelism**.
* Efficiently utilize the GPU's **SIMD-style execution units**.
* Enable lightweight and efficient scheduling using the **warp scheduler**.

## Thread Grouping into Warps


If a thread block contains 128 threads:

```text
128 threads / 32 threads per warp = 4 warps
```

* Threads 0–31 form warp 0.
* Threads 32–63 form warp 1.
* Threads 64–95 form warp 2.
* Threads 96–127 form warp 3.

## Execution Behavior


* **Threads in a warp execute in lockstep**, following the same instruction.
* If threads diverge in control flow (e.g., due to `if` conditions), **warp divergence** occurs, causing **serialized execution** and reduced performance.

## Warp Divergence


Warp divergence happens when threads in the same warp **follow different execution paths**.

Example:

```cpp
int tid = threadIdx.x;
if (tid % 2 == 0) {
    // Even threads do this
} else {
    // Odd threads do that
}
```

In this example, even and odd threads follow different branches, leading to warp divergence. This means that the warp will execute the even threads first,
then the odd threads, effectively serializing the execution and reducing performance.


Similarly it is a bad practice to have threads numbers that are not a multiple of 32 in a warp. This can lead to inefficient use of resources and increased
execution time.

For example, if a CUDA thread block has 30 threads, the last two threads in the warp will not be utilized, leading to wasted resources.
If a CUDA thread block has 40 threads, the block will be split as follows:

* Warp 0: threads 0–31
* Warp 1: threads 32–39 (only 8 threads)

So, the block will contain 2 warps, but the second warp is partially full (only 8 active threads).

* Warp 0 will execute normally with all 32 threads.
* Warp 1 will execute with only 8 active threads and the remaining 24 threads inactive (masked out).

The GPU still treats it as a full warp in terms of scheduling. Performance-wise, you're paying the cost of a full warp but using only part of it.


> **Important**
>
> * Write control flow to **minimize warp divergence**.
> * Ensure that threads access **contiguous memory** for coalesced memory access.
> * Aim for all threads in a warp to **follow the same path** for maximum efficiency.

## Some additional details


* Each SM is divided into processing partitions (also called sub-partitions).

* Each partition contains its own warp scheduler and instruction dispatch unit, along with a portion of the execution pipelines.

* On Volta (V100), an SM has 4 processing partitions, each with a warp scheduler, for a total of 4 schedulers per SM.

* An SM can host up to 64 resident warps (2048 threads).

* In each cycle, up to 4 warps can issue instructions (one per scheduler), while the rest wait.

* This deep pool of resident warps allows the GPU to quickly swap in ready warps when others are stalled (e.g., on memory or synchronization), ensuring high utilization and latency hiding.

![Processing Partitions](../docs/source/tutorial/figs/sm_partitions.png)

> **Key Points**
>
> 1. A warp is a group of 32 threads that execute the same instruction simultaneously.
> 1. Warps are fundamental to CUDA's execution model, enabling parallel processing.
> 1. Warp divergence can lead to performance issues; aim to minimize it.
> 1. Threads should be organized to maximize coalesced memory access and minimize divergence.
> 1. Understanding warps is essential for writing efficient CUDA kernels and optimizing performance.

## Exercise 04: Warps
**Time: 20 min**

Each of 256 threads acts on one pixel by recording `clock64()`. A single block contains eight warps and runs on one streaming multiprocessor (SM), so all readings use the same clock. We arrange the pixels as an **8 × 32 image**: each row is a warp, and each column is a lane within that warp.

The left image assigns a colour to each warp ID. The right image colours the measured timestamps: **blue means earlier, yellow means later**. These are enlarged logical pixels, rather than a photograph.

A block barrier brings all threads to the measurement point before any proceeds. Warps may then run concurrently or in any order; the barrier does not make all readings simultaneous. Similar colours can cross warp boundaries. This records when a thread reaches one instruction, not its full execution time or when a physical screen pixel changes.

Use a notebook kernel on an allocated GPU node with CUDA loaded. Plotting requires NumPy and Matplotlib.

Source: [NVIDIA CUDA clock documentation](https://docs.nvidia.com/cuda/cuda-programming-guide/05-appendices/cpp-language-support.html).

### Inspect the CUDA code

Look for the single-block launch, `__syncthreads()`, and the `clock64()` reading.

In [ ]:
!cat exercises/04-warps/warp_timing.cu

### Compile

The executable and measured data go into `warp_timing_output/`. Run this cell successfully before continuing.

In [ ]:
from pathlib import Path

output_dir = Path("warp_timing_output")
output_dir.mkdir(exist_ok=True)
!nvcc -O2 exercises/04-warps/warp_timing.cu -o warp_timing_output/warp_timing

### Record the timestamps

This writes actual GPU measurements to a CSV file. Rerun this cell and the plotting cell to compare different executions.

In [ ]:
# Remove the previous measurement so a failed run cannot show stale data.
(output_dir / "timestamps.csv").unlink(missing_ok=True)
!./warp_timing_output/warp_timing warp_timing_output/timestamps.csv

### Turn timestamps into colours

Subtract the earliest timestamp **before converting to floating point**, preserving small differences in the 64-bit clock readings. Each run uses its own colour scale, labelled in SM clock cycles after the earliest reading. Colours from different runs therefore do not represent the same absolute times.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm

data = np.genfromtxt(output_dir / "timestamps.csv", delimiter=",", names=True,
                     dtype=None, encoding="utf-8")
assert data.size == 256, "Expected one record for each of the 256 threads"
assert np.array_equal(data["thread"], np.arange(256))
assert np.array_equal(data["warp"], np.arange(256) // 32)
assert np.array_equal(data["lane"], np.arange(256) % 32)

clocks = data["clock"].astype(np.uint64)
cycles = (clocks - clocks.min()).astype(np.float64).reshape(8, 32)
warp_ids = data["warp"].reshape(8, 32)

fig, axes = plt.subplots(1, 2, figsize=(15, 5), constrained_layout=True)
warp_cmap = plt.get_cmap("tab10", 8)
warp_norm = BoundaryNorm(np.arange(-0.5, 8.5), warp_cmap.N)
left = axes[0].imshow(warp_ids, cmap=warp_cmap, norm=warp_norm,
                      interpolation="nearest", aspect="auto")
fig.colorbar(left, ax=axes[0], ticks=np.arange(8), label="Warp ID")
right = axes[1].imshow(cycles, cmap="cividis", vmin=0,
                       vmax=max(1.0, float(cycles.max())),
                       interpolation="nearest", aspect="auto")
fig.colorbar(right, ax=axes[1], label="SM clock cycles after earliest reading")
for ax, title in zip(axes, ["Which warp owns each pixel?", "When did each thread record its pixel?"]):
    ax.set(title=title, xlabel="Lane ID (column)", ylabel="Warp ID (row)",
           yticks=np.arange(8), xticks=[0, 7, 15, 23, 31])
    ax.set_yticks(np.arange(0.5, 7.5), minor=True)
    ax.grid(which="minor", axis="y", color="white", linewidth=0.5)
fig.savefig(output_dir / "warp_timing.png", dpi=160)
plt.show()
print(f"Timestamp spread: {int(cycles.max())} SM clock cycles")
print(f"Saved image: {output_dir / 'warp_timing.png'}")

### Explore the image

1. Do the 32 pixels in a row have similar timestamp colours? Do different rows ever share a similar colour?
2. Was warp 0 necessarily first? Use the numbers below rather than colour alone.
3. Rerun the measurement and plotting cells several times. Does the ordering change?

The observed order is a measurement, not a scheduling guarantee. Instrumentation itself affects scheduling, and this experiment does not measure warp divergence.

In [ ]:
print("Warp   earliest   latest   (SM clock cycles after first pixel)")
for warp, row in enumerate(cycles):
    print(f"{warp:4d}   {int(row.min()):8d}   {int(row.max()):6d}")